# Full fine-tuning Pythia-410M step50000 on the full cleaned Alpaca dataset

This Kaggle-ready notebook loads EleutherAI/pythia-410m at revision step50000, evaluates it unchanged, and then fully fine-tunes all model parameters for one epoch. It uses the entire usable yahma/alpaca-cleaned dataset: 256 shuffled examples are held out for validation and every remaining example is used for training.

> Enable a Kaggle GPU and Internet access before running all cells. This is an educational experiment, not a production chatbot.

## 1. Install dependencies

In [ ]:
%pip install -q -U \
    "transformers==4.57.1" \
    "datasets==4.4.1" \
    "accelerate==1.11.0" \
    "trl==0.26.2"

## 2. Imports, configuration, and GPU precision

In [ ]:
import math
import re
import shutil
from pathlib import Path

import accelerate
import datasets
import matplotlib.pyplot as plt
import pandas as pd
import torch
import transformers
import trl
from datasets import load_dataset
from IPython.display import display
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainerCallback, set_seed
from trl import SFTConfig, SFTTrainer

SEED = 42
MODEL_ID = "EleutherAI/pythia-410m"
MODEL_REVISION = "step50000"
DATASET_ID = "yahma/alpaca-cleaned"
OUTPUT_DIR = "/kaggle/working/pythia-410m-step50000-alpaca-runs"
FINAL_DIR = "/kaggle/working/pythia-410m-step50000-alpaca-final"
MAX_LENGTH = 384
VALIDATION_SIZE = 256
DATASET_NUM_PROC = 2

set_seed(SEED)
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"Datasets: {datasets.__version__}")
print(f"Accelerate: {accelerate.__version__}")
print(f"TRL: {trl.__version__}")

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU not found. In Kaggle, open Settings -> Accelerator, select a GPU, "
        "and restart the session."
    )

GPU_COUNT = torch.cuda.device_count()
capabilities = [torch.cuda.get_device_capability(i) for i in range(GPU_COUNT)]
AMPERE_OR_NEWER = all(major >= 8 for major, _ in capabilities)
USE_BF16 = AMPERE_OR_NEWER
MODEL_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
torch.backends.cuda.matmul.allow_tf32 = AMPERE_OR_NEWER
torch.backends.cudnn.allow_tf32 = AMPERE_OR_NEWER

print(f"CUDA version: {torch.version.cuda}")
for i in range(GPU_COUNT):
    props = torch.cuda.get_device_properties(i)
    major, minor = capabilities[i]
    print(f"GPU {i}: {props.name} | compute capability {major}.{minor}")
print(f"Selected training dtype: {MODEL_DTYPE}")
print(f"TF32 enabled: {AMPERE_OR_NEWER}")

## 3. Load the tokenizer and step50000 checkpoint

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=MODEL_DTYPE,
).to("cuda")

for config in (model.config, model.generation_config):
    config.pad_token_id = tokenizer.pad_token_id
    config.bos_token_id = tokenizer.bos_token_id
    config.eos_token_id = tokenizer.eos_token_id

print(f"Loaded: {MODEL_ID} @ {MODEL_REVISION}")
print(f"Model device: {next(model.parameters()).device}")
print(f"Model parameter dtype: {next(model.parameters()).dtype}")

## 4. Prepare the complete cleaned Alpaca dataset

Rows with empty outputs are removed and the remaining rows are shuffled deterministically. The first 256 rows become a fixed held-out validation set; all remaining usable rows become the training set. There is deliberately no TRAIN_SIZE limit.

In [ ]:
raw_dataset = load_dataset(DATASET_ID, split="train")
usable_dataset = raw_dataset.filter(
    lambda row: bool((row.get("output") or "").strip()),
    num_proc=DATASET_NUM_PROC,
)
if len(usable_dataset) <= VALIDATION_SIZE:
    raise ValueError(
        f"Need more than {VALIDATION_SIZE:,} usable examples, found {len(usable_dataset):,}."
    )

shuffled_dataset = usable_dataset.shuffle(seed=SEED)
validation_raw = shuffled_dataset.select(range(VALIDATION_SIZE))
train_raw = shuffled_dataset.select(range(VALIDATION_SIZE, len(shuffled_dataset)))
EOS_TOKEN = tokenizer.eos_token

def to_prompt_completion(row):
    instruction = (row.get("instruction") or "").strip()
    input_text = (row.get("input") or "").strip()
    if input_text:
        prompt = (
            "Below is an instruction that describes a task, paired with an input that provides "
            "further context. Write a response that appropriately completes the request.\n\n"
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            "### Response:\n"
        )
    else:
        prompt = (
            "Below is an instruction that describes a task. Write a response that appropriately "
            "completes the request.\n\n"
            f"### Instruction:\n{instruction}\n\n"
            "### Response:\n"
        )
    completion = (row.get("output") or "").strip() + EOS_TOKEN
    return {"prompt": prompt, "completion": completion}

train_dataset = train_raw.map(
    to_prompt_completion,
    remove_columns=train_raw.column_names,
    num_proc=DATASET_NUM_PROC,
)
validation_dataset = validation_raw.map(
    to_prompt_completion,
    remove_columns=validation_raw.column_names,
    num_proc=DATASET_NUM_PROC,
)

assert len(train_dataset) + len(validation_dataset) == len(usable_dataset)
assert len(validation_dataset) == VALIDATION_SIZE
assert all(text.endswith(EOS_TOKEN) for text in train_dataset["completion"])
print(f"Raw examples:        {len(raw_dataset):,}")
print(f"Usable examples:     {len(usable_dataset):,}")
print(f"Training examples:   {len(train_dataset):,}")
print(f"Validation examples: {len(validation_dataset):,}")

In [ ]:
example = train_dataset[0]
print("FULLY FORMATTED EXAMPLE")
print("-" * 80)
print(example["prompt"] + example["completion"])
print("-" * 80)
print(f"Completion ends with EOS: {example['completion'].endswith(EOS_TOKEN)}")

## 5. Configure full-parameter supervised fine-tuning

Loss is calculated only on completion and EOS tokens. Prompt tokens remain visible as context but are masked from the loss. No LoRA or PEFT adapter is used.

In [ ]:
test_questions = [
    {"question": "Explain photosynthesis in exactly two sentences.", "input": ""},
    {"question": "Translate 'The weather is beautiful today' into French.", "input": ""},
    {"question": "Calculate the average speed for 180 km in 3 hours.", "input": ""},
    {"question": "Write a Python function that checks whether an integer is prime.", "input": ""},
    {"question": "Give exactly three concise job interview tips.", "input": ""},
    {"question": "Explain why leaves appear green in simple language.", "input": ""},
]

def format_test_prompt(question, input_text=""):
    row = {"instruction": question, "input": input_text, "output": "placeholder"}
    return to_prompt_completion(row)["prompt"]

class LossPrinterCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        logs = logs or {}
        if state.global_step > 0 and state.global_step % 10 == 0 and "loss" in logs:
            grad_norm = logs.get("grad_norm")
            grad_text = "n/a" if grad_norm is None else f"{float(grad_norm):.4f}"
            print(
                f"step {state.global_step:5d} | loss: {float(logs['loss']):.4f} | "
                f"lr: {float(logs.get('learning_rate', 0.0)):.6f} | grad_norm: {grad_text}"
            )

In [ ]:
model.config.use_cache = False
sft_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    max_grad_norm=1.0,
    optim="adamw_torch",
    max_length=MAX_LENGTH,
    completion_only_loss=True,
    packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=not USE_BF16,
    bf16=USE_BF16,
    tf32=AMPERE_OR_NEWER,
    logging_strategy="steps",
    logging_steps=10,
    logging_first_step=True,
    eval_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=250,
    save_total_limit=2,
    report_to="none",
    dataset_num_proc=DATASET_NUM_PROC,
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
    callbacks=[LossPrinterCallback()],
)

total_parameters = sum(parameter.numel() for parameter in model.parameters())
trainable_parameters = sum(
    parameter.numel() for parameter in model.parameters() if parameter.requires_grad
)
trainable_percentage = 100 * trainable_parameters / total_parameters
effective_batch_size = (
    sft_args.per_device_train_batch_size
    * sft_args.gradient_accumulation_steps
    * GPU_COUNT
)
estimated_optimizer_steps = math.ceil(len(train_dataset) / effective_batch_size)

print(f"Total parameters: {total_parameters:,}")
print(f"Trainable parameters: {trainable_parameters:,} ({trainable_percentage:.2f}%)")
if trainable_percentage < 99.0:
    raise RuntimeError("Fewer than 99% of parameters are trainable.")
print(f"Effective batch size: {effective_batch_size}")
print(f"Estimated optimizer steps for one epoch: {estimated_optimizer_steps:,}")

## 6. Evaluate the untouched checkpoint

In [ ]:
def generate_answers(current_model, questions):
    current_model.eval()
    device = next(current_model.parameters()).device
    answers = []
    for item in questions:
        prompt = format_test_prompt(item["question"], item.get("input", ""))
        encoded = tokenizer(prompt, return_tensors="pt")
        encoded = {name: tensor.to(device) for name, tensor in encoded.items()}
        prompt_length = encoded["input_ids"].shape[1]
        with torch.inference_mode():
            generated = current_model.generate(
                **encoded,
                do_sample=False,
                max_new_tokens=128,
                repetition_penalty=1.05,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
            )
        new_tokens = generated[0, prompt_length:]
        answers.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())
    return answers

baseline_metrics = trainer.evaluate()
baseline_eval_loss = float(baseline_metrics["eval_loss"])
baseline_answers = generate_answers(model, test_questions)
print(f"Baseline validation loss: {baseline_eval_loss:.6f}")
for item, answer in zip(test_questions, baseline_answers):
    print(f"QUESTION: {item['question']}")
    print(f"BEFORE: {answer}\n")

## 7. Train for one epoch

The model is promoted to FP32 master weights immediately before optimization. Forward and backward computation still use the selected mixed precision.

In [ ]:
trainer.model.float()
torch.cuda.empty_cache()
print(
    f"Master-weight dtype: {next(trainer.model.parameters()).dtype}; "
    f"AMP compute dtype: {MODEL_DTYPE}"
)
train_output = trainer.train()
print(train_output)
training_log_history = list(trainer.state.log_history)

## 8. Inspect training and validation loss

In [ ]:
log_df = pd.DataFrame(training_log_history)
requested_columns = [
    "step", "loss", "eval_loss", "learning_rate", "grad_norm",
    "entropy", "mean_token_accuracy", "num_tokens",
]
available_columns = [column for column in requested_columns if column in log_df.columns]
display(log_df[available_columns].dropna(how="all").reset_index(drop=True))

train_rows = log_df.dropna(subset=["loss"]) if "loss" in log_df else pd.DataFrame()
eval_rows = log_df.dropna(subset=["eval_loss"]) if "eval_loss" in log_df else pd.DataFrame()
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
if not train_rows.empty:
    axes[0].plot(train_rows["step"], train_rows["loss"], linewidth=1.5)
else:
    axes[0].text(0.5, 0.5, "No training-loss rows found", ha="center")
axes[0].set(title="Training loss", xlabel="Optimizer step", ylabel="Loss")
axes[0].grid(alpha=0.25)

if not eval_rows.empty:
    axes[1].plot(eval_rows["step"], eval_rows["eval_loss"], marker="o", label="Validation")
axes[1].axhline(baseline_eval_loss, color="tab:red", linestyle="--", label="Baseline")
axes[1].set(title="Fixed validation loss", xlabel="Optimizer step", ylabel="Loss")
axes[1].grid(alpha=0.25)
axes[1].legend()
plt.tight_layout()
plt.show()

## 9. Final evaluation and before/after generations

In [ ]:
final_metrics = trainer.evaluate()
final_eval_loss = float(final_metrics["eval_loss"])
absolute_improvement = baseline_eval_loss - final_eval_loss
percentage_improvement = (
    100 * absolute_improvement / baseline_eval_loss
    if math.isfinite(baseline_eval_loss) and baseline_eval_loss != 0
    else float("nan")
)

def safe_perplexity(loss):
    if not math.isfinite(loss) or loss > 700:
        return float("inf")
    return math.exp(loss)

print(f"Baseline validation loss: {baseline_eval_loss:.6f}")
print(f"Final validation loss:    {final_eval_loss:.6f}")
print(f"Absolute improvement:     {absolute_improvement:.6f}")
print(f"Percentage improvement:   {percentage_improvement:.2f}%")
print(f"Baseline perplexity:      {safe_perplexity(baseline_eval_loss):.4f}")
print(f"Final perplexity:         {safe_perplexity(final_eval_loss):.4f}")

In [ ]:
trainer.model.gradient_checkpointing_disable()
trainer.model.to(dtype=MODEL_DTYPE)
trainer.model.config.use_cache = True
trainer.model.generation_config.use_cache = True
trainer.model.eval()

finetuned_answers = generate_answers(trainer.model, test_questions)
comparison_df = pd.DataFrame(
    {
        "question": [item["question"] for item in test_questions],
        "optional_input": [item.get("input", "") for item in test_questions],
        "before_training": baseline_answers,
        "after_training": finetuned_answers,
    }
)
display(comparison_df)

## 10. Save the complete fine-tuned model and results

In [ ]:
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

final_path = Path(FINAL_DIR)
comparison_path = final_path / "comparison_results.csv"
comparison_df.to_csv(comparison_path, index=False)
archive_path = shutil.make_archive(
    str(final_path),
    "zip",
    root_dir=final_path.parent,
    base_dir=final_path.name,
)

print("Saved full-model files:")
for file_path in sorted(path for path in final_path.rglob("*") if path.is_file()):
    size_mib = file_path.stat().st_size / 1024**2
    print(f"  {file_path.relative_to(final_path)} - {size_mib:.2f} MiB")
archive_size_mib = Path(archive_path).stat().st_size / 1024**2
print(f"ZIP archive: {archive_path} - {archive_size_mib:.2f} MiB")
print("This archive contains a complete fine-tuned model, not a LoRA adapter.")

## 11. Memory fallback

If the configured run raises a CUDA out-of-memory error, restart the session and change only these settings: per_device_train_batch_size=1, gradient_accumulation_steps=16, and MAX_LENGTH=256. The single-GPU effective batch size remains 16.